# 02 · Preprocesamiento y Feature Engineering
## Proyecto: Estimador de Capacidad de Pago — Sector Informal (ENIGH 2024)

Este notebook toma las decisiones documentadas en el EDA (`01_eda_concentradohogar.ipynb`)
y construye el dataset listo para modelar (`df_ml`), que se guarda en
`data/processed/`.

**Decisiones aplicadas (acordadas previamente):**

1. **Variable objetivo — se construyen DOS versiones**, para comparar en el
   artículo:
   - `clase_ingreso_15k`: umbral de $15,000 MXN/mes (≈ 2 salarios mínimos
     generales 2024).
   - `clase_ingreso_mediana`: umbral igual a la mediana muestral de
     `ing_mensual` (≈ $18,555 MXN, prácticamente igual a la Línea de
     Pobreza por Ingresos urbana de CONEVAL para un hogar de 4 integrantes,
     agosto 2024: $18,259.84). Esta versión produce un balance de clases
     ~50/50.
2. **Tasa de dependencia** = `tot_integ` / `ocupados` (si `ocupados == 0`,
   se usa `tot_integ`).
3. **Región geográfica corregida**: se usa la regionalización de 4 regiones
   del Banco de México (Norte, Centro-Norte, Centro, Sur), que cubre las
   **32 entidades** sin categoría "Otro" residual.
4. **`est_socio`** se conserva como variable numérica ordinal (1=Bajo a
   4=Alto) — el notebook de modelado decidirá si se incluye o no
   (análisis con/sin).
5. **Variables binarias/ordinales**: `nivel_urbano` (tam_loc invertido),
   `educa_num`, `es_mujer_jefa`.
6. **One-Hot Encoding** para `region` y `clase_hog` (con `drop_first=True`).


## 0. Configuración inicial y carga de datos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_RAW / "conjunto_de_datos_concentradohogar_enigh2024_ns.csv"


In [2]:
# Solo cargamos las columnas necesarias (optimiza memoria)
COLUMNAS_INTERES = [
    "folioviv", "foliohog",
    "ubica_geo", "tam_loc", "est_socio",
    "clase_hog", "sexo_jefe", "edad_jefe", "educa_jefe",
    "tot_integ", "ocupados",
    "factor", "ing_cor",
]

DTYPE_CATALOGOS = {
    "folioviv": str,
    "foliohog": str,
    "ubica_geo": str,
    "tam_loc": str,
    "est_socio": str,
    "clase_hog": str,
    "sexo_jefe": str,
    "educa_jefe": str,
}

df = pd.read_csv(DATA_FILE, usecols=COLUMNAS_INTERES, dtype=DTYPE_CATALOGOS)
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
df.head()


Filas: 91,414  |  Columnas: 13


,folioviv,foliohog,ubica_geo,tam_loc,est_socio,factor,clase_hog,sexo_jefe,edad_jefe,educa_jefe,tot_integ,ocupados,ing_cor
0,0100001901,1,01001,1,3,207,2,1,32,06,4,2,138232.38
1,0100001902,1,01001,1,3,207,2,1,48,09,4,2,118014.04
2,0100001904,1,01001,1,3,207,2,2,60,06,2,2,46866.32
3,0100001905,1,01001,1,3,207,2,1,43,08,4,3,110430.10
4,0100002501,1,01001,1,2,196,2,2,29,08,4,2,99494.12


## 1. Variable objetivo: dos versiones de `clase_ingreso`

`ing_cor` es trimestral → se convierte a mensual dividiendo entre 3.

In [3]:
df["ing_mensual"] = df["ing_cor"] / 3

# --- Umbral 1: $15,000 MXN/mes (~2 salarios mínimos generales 2024) ---
UMBRAL_15K = 15000.0

# --- Umbral 2: mediana muestral (~ LPI urbana CONEVAL hogar 4 integrantes) ---
UMBRAL_MEDIANA = df["ing_mensual"].median()

df["clase_ingreso_15k"] = (df["ing_mensual"] >= UMBRAL_15K).astype(int)
df["clase_ingreso_mediana"] = (df["ing_mensual"] >= UMBRAL_MEDIANA).astype(int)

print(f"Umbral 1 (~2 salarios mínimos): ${UMBRAL_15K:,.2f}")
print(f"Umbral 2 (mediana muestral):    ${UMBRAL_MEDIANA:,.2f}")
print()

for col in ["clase_ingreso_15k", "clase_ingreso_mediana"]:
    print(f"--- Balance de clases: {col} ---")
    print((df[col].value_counts(normalize=True) * 100).round(2).rename("pct"))
    print()


Umbral 1 (~2 salarios mínimos): $15,000.00
Umbral 2 (mediana muestral):    $18,555.39

--- Balance de clases: clase_ingreso_15k ---
clase_ingreso_15k
1    61.86
0    38.14
Name: pct, dtype: float64

--- Balance de clases: clase_ingreso_mediana ---
clase_ingreso_mediana
1    50.01
0    49.99
Name: pct, dtype: float64



## 2. Tasa de dependencia económica

$$
\text{tasa\_dependencia} =
\begin{cases}
\dfrac{\text{tot\_integ}}{\text{ocupados}} & \text{si ocupados} > 0 \\
\text{tot\_integ} & \text{si ocupados} = 0
\end{cases}
$$

Representa cuántos integrantes del hogar "dependen" de cada persona ocupada.
Un hogar sin ocupados se interpreta como que todos sus integrantes dependen
de fuentes no laborales (ahorros, transferencias, jubilaciones, etc.).

In [4]:
df["tasa_dependencia"] = np.where(
    df["ocupados"] > 0,
    df["tot_integ"] / df["ocupados"],
    df["tot_integ"]
)

df["tasa_dependencia"].describe().round(2)


count    91414.00
mean         2.15
std          1.19
min          1.00
25%          1.00
50%          2.00
75%          3.00
max         11.00
Name: tasa_dependencia, dtype: float64

## 3. Variables del jefe(a) de hogar y de la vivienda

- `nivel_urbano`: se invierte `tam_loc` (1=metrópoli grande → 4=rural) para
  que un valor **mayor** signifique **mayor urbanización**: `nivel_urbano = 5 - tam_loc`.
- `educa_num`: se convierte `educa_jefe` (código de catálogo "01"-"11") a
  entero, preservando el orden (Sin instrucción=1 → Posgrado=11).
- `es_mujer_jefa`: binaria, 1 si `sexo_jefe == "2"` (Mujer).
- `est_socio_num`: se conserva `est_socio` como entero ordinal (1=Bajo a
  4=Alto). Se evalúa más adelante si se incluye o no como predictor.

In [5]:
df["nivel_urbano"] = 5 - df["tam_loc"].astype(int)
df["educa_num"] = df["educa_jefe"].astype(int)
df["es_mujer_jefa"] = (df["sexo_jefe"] == "2").astype(int)
df["est_socio_num"] = df["est_socio"].astype(int)

df[["tam_loc", "nivel_urbano", "educa_jefe", "educa_num",
    "sexo_jefe", "es_mujer_jefa", "est_socio", "est_socio_num"]].head()


,tam_loc,nivel_urbano,educa_jefe,educa_num,sexo_jefe,es_mujer_jefa,est_socio,est_socio_num
0,1,4,06,6,1,0,3,3
1,1,4,09,9,1,0,3,3
2,1,4,06,6,2,1,3,3
3,1,4,08,8,1,0,3,3
4,1,4,08,8,2,1,2,2


## 4. Región geográfica (regionalización Banxico — 4 regiones, 32 estados)

Se utiliza la regionalización de **4 regiones económicas** del Banco de
México, que agrupa las 32 entidades federativas sin dejar ninguna fuera
("Norte", "Centro-Norte", "Centro" y "Sur"):

- **Norte**: Baja California, Coahuila, Chihuahua, Nuevo León, Sonora, Tamaulipas
- **Centro-Norte**: Aguascalientes, Baja California Sur, Colima, Durango,
  Jalisco, Michoacán, Nayarit, San Luis Potosí, Sinaloa, Zacatecas
- **Centro**: CDMX, Estado de México, Guanajuato, Hidalgo, Morelos, Puebla,
  Querétaro, Tlaxcala
- **Sur**: Campeche, Chiapas, Guerrero, Oaxaca, Quintana Roo, Tabasco,
  Veracruz, Yucatán

A diferencia del mapeo de 5 macro-regiones usado en la primera versión del
proyecto (que dejaba 2 estados — Colima y Durango — en una categoría
residual "Otro"), esta regionalización cubre las 32 entidades sin
categoría residual, y tiene respaldo en los Reportes sobre las Economías
Regionales del Banco de México.

In [6]:
REGION_MAP = {
    # Norte (6)
    "02": "Norte", "05": "Norte", "08": "Norte",
    "19": "Norte", "26": "Norte", "28": "Norte",
    # Centro-Norte (10)
    "01": "Centro_Norte", "03": "Centro_Norte", "06": "Centro_Norte",
    "10": "Centro_Norte", "14": "Centro_Norte", "16": "Centro_Norte",
    "18": "Centro_Norte", "24": "Centro_Norte", "25": "Centro_Norte",
    "32": "Centro_Norte",
    # Centro (8)
    "09": "Centro", "11": "Centro", "13": "Centro", "15": "Centro",
    "17": "Centro", "21": "Centro", "22": "Centro", "29": "Centro",
    # Sur (8)
    "04": "Sur", "07": "Sur", "12": "Sur", "20": "Sur",
    "23": "Sur", "27": "Sur", "30": "Sur", "31": "Sur",
}

df["estado_id"] = df["ubica_geo"].str[:2]
df["region"] = df["estado_id"].map(REGION_MAP)

# Verificación de cobertura: no debe haber valores nulos en 'region'
n_sin_region = df["region"].isnull().sum()
print(f"Estados no mapeados a una región: {n_sin_region}")
assert n_sin_region == 0, "Hay códigos de estado sin region asignada — revisar REGION_MAP"

print("\nDistribución de hogares por región:")
print(df["region"].value_counts())


Estados no mapeados a una región: 0

Distribución de hogares por región:
region
Centro_Norte    27359
Centro          22266
Norte           21698
Sur             20091
Name: count, dtype: int64


## 5. One-Hot Encoding de variables categóricas nominales

Se aplica `pd.get_dummies` con `drop_first=True` a `region` y `clase_hog`
para evitar multicolinealidad perfecta (trampa de variables ficticias).

In [7]:
df_encoded = pd.get_dummies(df, columns=["region", "clase_hog"], drop_first=True)

# Las columnas dummy son booleanas; las convertimos a enteros (0/1) para
# compatibilidad con todos los modelos de scikit-learn
dummy_cols = [c for c in df_encoded.columns if c.startswith("region_") or c.startswith("clase_hog_")]
df_encoded[dummy_cols] = df_encoded[dummy_cols].astype(int)

print("Columnas dummy creadas:")
print(dummy_cols)


Columnas dummy creadas:
['region_Centro_Norte', 'region_Norte', 'region_Sur', 'clase_hog_2', 'clase_hog_3', 'clase_hog_4', 'clase_hog_5']


## 6. Construcción del dataset final (`df_ml`)

Se conservan:
- Identificadores (`folioviv`, `foliohog`) — útiles para trazabilidad, se
  excluyen de `X` en el modelado.
- **Dos variables objetivo**: `clase_ingreso_15k`, `clase_ingreso_mediana`.
- Predictoras: `edad_jefe`, `tasa_dependencia`, `nivel_urbano`, `educa_num`,
  `es_mujer_jefa`, `est_socio_num`, dummies de `region_*` y `clase_hog_*`.

Se descartan las columnas originales ya transformadas (`ubica_geo`,
`tam_loc`, `educa_jefe`, `sexo_jefe`, `estado_id`, `ing_cor`, `ing_mensual`,
`tot_integ`, `ocupados`, `factor`, `est_socio`) para evitar fuga de
información y redundancia.

> Nota: `ing_mensual` se descarta del dataset de modelado porque es la base
> directa de la variable objetivo (fuga de información). Se conserva
> únicamente como variable auxiliar durante el preprocesamiento.

In [8]:
COLUMNAS_DESCARTE = [
    "ubica_geo", "tam_loc", "educa_jefe", "sexo_jefe", "estado_id",
    "ing_cor", "ing_mensual", "tot_integ", "ocupados", "factor", "est_socio",
]

df_ml = df_encoded.drop(columns=COLUMNAS_DESCARTE)

print("Forma del dataset final:", df_ml.shape)
df_ml.head()


Forma del dataset final: (91414, 17)


,folioviv,foliohog,edad_jefe,clase_ingreso_15k,clase_ingreso_mediana,tasa_dependencia,nivel_urbano,educa_num,es_mujer_jefa,est_socio_num,region_Centro_Norte,region_Norte,region_Sur,clase_hog_2,clase_hog_3,clase_hog_4,clase_hog_5
0,0100001901,1,32,1,1,2.000000,4,6,0,3,1,0,0,1,0,0,0
1,0100001902,1,48,1,1,2.000000,4,9,0,3,1,0,0,1,0,0,0
2,0100001904,1,60,1,0,1.000000,4,6,1,3,1,0,0,1,0,0,0
3,0100001905,1,43,1,1,1.333333,4,8,0,3,1,0,0,1,0,0,0
4,0100002501,1,29,1,1,2.000000,4,8,1,2,1,0,0,1,0,0,0


In [9]:
print("Tipos de dato del dataset final:")
print(df_ml.dtypes)

print("\nValores nulos en df_ml:")
print(df_ml.isnull().sum().sum(), "valores nulos en total")


Tipos de dato del dataset final:
folioviv                  object
foliohog                  object
edad_jefe                  int64
clase_ingreso_15k          int64
clase_ingreso_mediana      int64
tasa_dependencia         float64
nivel_urbano               int64
educa_num                  int64
es_mujer_jefa              int64
est_socio_num              int64
region_Centro_Norte        int64
region_Norte               int64
region_Sur                 int64
clase_hog_2                int64
clase_hog_3                int64
clase_hog_4                int64
clase_hog_5                int64
dtype: object

Valores nulos en df_ml:
0 valores nulos en total


## 7. Diccionario de variables del dataset procesado

Tabla de referencia para documentar `df_ml` en el artículo.

In [10]:
diccionario_df_ml = pd.DataFrame([
    ("folioviv", "Identificador de vivienda (no usar como predictor)", "Texto"),
    ("foliohog", "Identificador de hogar (no usar como predictor)", "Texto"),
    ("clase_ingreso_15k", "Target 1: 1 = ing. mensual >= $15,000 (~2 SM)", "Binaria"),
    ("clase_ingreso_mediana", "Target 2: 1 = ing. mensual >= mediana muestral (~LPI urbana CONEVAL)", "Binaria"),
    ("edad_jefe", "Edad del jefe(a) de hogar", "Numérica continua"),
    ("tasa_dependencia", "tot_integ / ocupados (o tot_integ si ocupados=0)", "Numérica continua"),
    ("nivel_urbano", "5 - tam_loc (mayor = más urbano)", "Ordinal (1-4)"),
    ("educa_num", "Nivel educativo del jefe(a) (1=Sin instrucción .. 11=Posgrado)", "Ordinal (1-11)"),
    ("es_mujer_jefa", "1 si el jefe(a) de hogar es mujer", "Binaria"),
    ("est_socio_num", "Estrato socioeconómico INEGI (1=Bajo .. 4=Alto)", "Ordinal (1-4)"),
    ("region_*", "Dummies de región (Centro_Norte, Norte, Sur); base = Centro", "Binaria (One-Hot)"),
    ("clase_hog_*", "Dummies de tipo de hogar; base = Unipersonal (1)", "Binaria (One-Hot)"),
], columns=["variable", "descripcion", "tipo"])

diccionario_df_ml


,variable,descripcion,tipo
0,folioviv,Identificador de vivienda (no usar como predic...,Texto
1,foliohog,Identificador de hogar (no usar como predictor),Texto
2,clase_ingreso_15k,"Target 1: 1 = ing. mensual >= $15,000 (~2 SM)",Binaria
3,clase_ingreso_mediana,Target 2: 1 = ing. mensual >= mediana muestral...,Binaria
4,edad_jefe,Edad del jefe(a) de hogar,Numérica continua
5,tasa_dependencia,tot_integ / ocupados (o tot_integ si ocupados=0),Numérica continua
6,nivel_urbano,5 - tam_loc (mayor = más urbano),Ordinal (1-4)
7,educa_num,Nivel educativo del jefe(a) (1=Sin instrucción...,Ordinal (1-11)
8,es_mujer_jefa,1 si el jefe(a) de hogar es mujer,Binaria
9,est_socio_num,Estrato socioeconómico INEGI (1=Bajo .. 4=Alto),Ordinal (1-4)


## 8. Guardar dataset procesado

Se guarda como CSV en `data/processed/` para usarlo en el notebook de
modelado (`03_modelado.ipynb`).

In [11]:
OUTPUT_FILE = DATA_PROCESSED / "concentradohogar_procesado.csv"
df_ml.to_csv(OUTPUT_FILE, index=False)

# También se guarda el diccionario de variables como referencia
diccionario_df_ml.to_csv(DATA_PROCESSED / "diccionario_df_ml.csv", index=False)

print(f"✅ Dataset procesado guardado en: {OUTPUT_FILE}")
print(f"   Filas: {df_ml.shape[0]:,}  |  Columnas: {df_ml.shape[1]}")


✅ Dataset procesado guardado en: ..\data\processed\concentradohogar_procesado.csv
   Filas: 91,414  |  Columnas: 17


## Resumen de decisiones aplicadas

| Decisión | Resolución |
|---|---|
| Umbral de `clase_ingreso` | Se generan **dos** versiones: `clase_ingreso_15k` y `clase_ingreso_mediana`, para comparar en el artículo |
| Tasa de dependencia | `tot_integ / ocupados` (o `tot_integ` si `ocupados=0`) |
| Región geográfica | Regionalización Banxico de 4 regiones, cobertura completa de 32 estados |
| `est_socio` | Conservada como `est_socio_num` (ordinal 1-4); se decide en el notebook de modelado si se incluye o no |
| Variables binarias/ordinales | `nivel_urbano`, `educa_num`, `es_mujer_jefa` |
| One-Hot Encoding | `region` y `clase_hog`, con `drop_first=True` |

**Siguiente paso:** `03_modelado.ipynb` — entrenamiento y evaluación de
Regresión Logística, Red Neuronal (MLP) y Random Forest, con validación
cruzada, sobre ambas variables objetivo y con/sin `est_socio_num` y
`class_weight='balanced'`.
